# Wind Resources and Wind Offshore Power Generation on the German Coast

**Project:** wind_energy_germancoast  
**Notebook:** `01_eda.ipynb` — Cleaning, Merging, EDA & Correlation Analysis  
**Data sources:**
- SMARD (Bundesnetzagentur) — Offshore wind power generation, aggregated Germany (MWh/day)
- DWD (Deutscher Wetterdienst) — Daily wind speed at coastal stations (m/s)

**Period:** 2021–2025  
**Geography:** German coast — North Sea (Sylt/List, Helgoland) + Baltic Sea (Fehmarn)

---
## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## 2. Load Data

### 2.1 SMARD — Offshore Wind Power Generation

**Source:** SMARD.de (Bundesnetzagentur)  
**Variable:** `wind_offshore_mwh` — total daily generation (MWh) of all German offshore wind farms (North Sea + Baltic Sea)  
**Note:** Aggregated across all four German TSO control areas.

In [2]:
smard = pd.read_csv('../data/raw/smard_wind_offshore_2021_2025.csv', parse_dates=['date'])

print(f'Shape: {smard.shape}')
print(f'Date range: {smard["date"].min()} → {smard["date"].max()}')
print()
smard.head()

Shape: (1825, 2)
Date range: 2021-01-01 00:00:00 → 2025-12-30 00:00:00



,date,wind_offshore_mwh
0,2021-01-01,17009.50
1,2021-01-02,102594.25
2,2021-01-03,121213.50
3,2021-01-04,139903.25
4,2021-01-05,87956.75


### 2.2 DWD — Coastal Wind Speed Stations

**Source:** DWD (Deutscher Wetterdienst)  
**Variable:** `wind_speed_ms` — mean daily wind speed (m/s)  
**Stations:**
- **List auf Sylt** — North Sea (northwest)
- **Helgoland** — North Sea (offshore island, closest to wind farms)
- **Fehmarn** — Baltic Sea (east, proxy for Baltic offshore generation)

In [3]:
dwd = pd.read_csv('../data/raw/dwd_wind_stations_2021_2025.csv', parse_dates=['date'])

print(f'Shape: {dwd.shape}')
print(f'Date range: {dwd["date"].min()} → {dwd["date"].max()}')
print(f'Stations : {dwd["station_name"].unique()}')
print(f'Seas     : {dwd["sea"].unique()}')
print()
dwd.head(10)

Shape: (5473, 5)
Date range: 2021-01-01 00:00:00 → 2025-12-31 00:00:00
Stations : ['Fehmarn' 'Helgoland' 'List auf Sylt']
Seas     : ['Baltic Sea' 'North Sea']



,date,station_id,station_name,sea,wind_speed_ms
0,2021-01-01,1503,Fehmarn,Baltic Sea,2.70
1,2021-01-02,1503,Fehmarn,Baltic Sea,2.80
2,2021-01-03,1503,Fehmarn,Baltic Sea,4.40
3,2021-01-04,1503,Fehmarn,Baltic Sea,6.40
4,2021-01-05,1503,Fehmarn,Baltic Sea,4.70
5,2021-01-06,1503,Fehmarn,Baltic Sea,3.10
6,2021-01-07,1503,Fehmarn,Baltic Sea,2.90
7,2021-01-08,1503,Fehmarn,Baltic Sea,2.20
8,2021-01-09,1503,Fehmarn,Baltic Sea,1.30
9,2021-01-10,1503,Fehmarn,Baltic Sea,3.70


---
## 3. Cleaning

For each dataset we check:
1. Data types (`dtypes`)
2. Null values
3. Date range — consistency and continuity
4. Values ​​outside the range (obvious outliers)

### 3.1 SMARD — Cleaning

In [4]:
# --- Dtypes ---
print('=== DTYPES ===')
print(smard.dtypes)
print()

=== DTYPES ===
date                 datetime64[ns]
wind_offshore_mwh           float64
dtype: object



In [5]:
# --- Null values ---
print('=== NULL VALUES ===')
null_counts = smard.isnull().sum()
null_pct = (smard.isnull().sum() / len(smard) * 100).round(2)
print(pd.DataFrame({'nulls': null_counts, '%': null_pct}))
print()

=== NULL VALUES ===
                   nulls    %
date                   0 0.00
wind_offshore_mwh      0 0.00



In [6]:
# --- Date range & continuity ---
print('=== DATE RANGE ===')
print(f'First date : {smard["date"].min()}')
print(f'Last date  : {smard["date"].max()}')
print(f'Total rows : {len(smard)}')

expected_days = (smard['date'].max() - smard['date'].min()).days + 1
missing_days = expected_days - len(smard)
print(f'Expected days : {expected_days}')
print(f'Missing days  : {missing_days}')
print()

=== DATE RANGE ===
First date : 2021-01-01 00:00:00
Last date  : 2025-12-30 00:00:00
Total rows : 1825
Expected days : 1825
Missing days  : 0



In [7]:
# --- Show which dates are missing (if any) ---
full_range = pd.date_range(start=smard['date'].min(), end=smard['date'].max(), freq='D')
missing_dates_smard = full_range.difference(smard['date'])

if len(missing_dates_smard) == 0:
    print('No missing dates in SMARD. ✓')
else:
    print(f'{len(missing_dates_smard)} missing dates:')
    print(missing_dates_smard)

No missing dates in SMARD. ✓


In [8]:
# --- Basic stats & outlier check ---
print('=== DESCRIPTIVE STATS ===')
print(smard['wind_offshore_mwh'].describe())
print()

# Negative values (physically impossible)
neg = smard[smard['wind_offshore_mwh'] < 0]
print(f'Negative values: {len(neg)}')

# Zero values (possible during curtailment but suspicious)
zeros = smard[smard['wind_offshore_mwh'] == 0]
print(f'Zero values: {len(zeros)}')

=== DESCRIPTIVE STATS ===
count     1825.00
mean     68014.61
std      40290.78
min        179.00
25%      32865.50
50%      65361.25
75%     101503.25
max     175514.42
Name: wind_offshore_mwh, dtype: float64

Negative values: 0
Zero values: 0


In [9]:
# --- Duplicate dates ---
dupes = smard[smard.duplicated(subset='date', keep=False)]
print(f'Duplicate dates in SMARD: {len(dupes)}')
if len(dupes) > 0:
    print(dupes)

Duplicate dates in SMARD: 0


### 3.2 DWD — Cleaning

In [10]:
# --- Dtypes ---
print('=== DTYPES ===')
print(dwd.dtypes)
print()

=== DTYPES ===
date             datetime64[ns]
station_id                int64
station_name             object
sea                      object
wind_speed_ms           float64
dtype: object



In [11]:
# --- Null values per station ---
print('=== NULL VALUES PER STATION ===')
null_by_station = dwd.groupby('station_name')['wind_speed_ms'].apply(
    lambda x: pd.Series({
        'nulls': x.isnull().sum(),
        '%': round(x.isnull().sum() / len(x) * 100, 2)
    })
).unstack()
print(null_by_station)
print()

=== NULL VALUES PER STATION ===
               nulls    %
station_name             
Fehmarn        33.00 1.81
Helgoland       2.00 0.11
List auf Sylt  93.00 5.09



In [12]:
# --- Date range & continuity per station ---
print('=== DATE RANGE PER STATION ===')
date_summary = dwd.groupby('station_name')['date'].agg(['min', 'max', 'count'])
date_summary.columns = ['first_date', 'last_date', 'n_records']
print(date_summary)
print()

=== DATE RANGE PER STATION ===
              first_date  last_date  n_records
station_name                                  
Fehmarn       2021-01-01 2025-12-31       1821
Helgoland     2021-01-01 2025-12-31       1826
List auf Sylt 2021-01-01 2025-12-31       1826



In [13]:
# --- Missing dates per station ---
print('=== MISSING DATES PER STATION ===')
for station, group in dwd.groupby('station_name'):
    full_range = pd.date_range(start=group['date'].min(), end=group['date'].max(), freq='D')
    missing = full_range.difference(group['date'])
    print(f'{station}: {len(missing)} missing dates')
    if len(missing) > 0 and len(missing) <= 10:
        print(f'  → {list(missing.date)}')

=== MISSING DATES PER STATION ===
Fehmarn: 5 missing dates
  → [datetime.date(2024, 4, 24), datetime.date(2024, 4, 25), datetime.date(2024, 4, 26), datetime.date(2024, 4, 27), datetime.date(2024, 4, 28)]
Helgoland: 0 missing dates
List auf Sylt: 0 missing dates


In [14]:
# --- Basic stats per station ---
print('=== DESCRIPTIVE STATS PER STATION ===')
print(dwd.groupby('station_name')['wind_speed_ms'].describe().round(2))
print()

# Stats per sea (North Sea vs Baltic)
print('=== DESCRIPTIVE STATS PER SEA ===')
print(dwd.groupby('sea')['wind_speed_ms'].describe().round(2))
print()

# Negative wind speed (physically impossible)
neg_dwd = dwd[dwd['wind_speed_ms'] < 0]
print(f'Negative wind speed values: {len(neg_dwd)}')


=== DESCRIPTIVE STATS PER STATION ===
                count  mean  std  min  25%  50%  75%   max
station_name                                              
Fehmarn       1788.00  3.70 1.74 0.60 2.50 3.30 4.70 13.10
Helgoland     1824.00  4.21 1.69 1.20 3.00 3.90 5.12 12.90
List auf Sylt 1733.00  7.25 2.70 1.50 5.10 6.90 9.00 16.60

=== DESCRIPTIVE STATS PER SEA ===
             count  mean  std  min  25%  50%  75%   max
sea                                                    
Baltic Sea 1788.00  3.70 1.74 0.60 2.50 3.30 4.70 13.10
North Sea  3557.00  5.69 2.71 1.20 3.60 5.10 7.30 16.60

Negative wind speed values: 0


In [15]:
# --- Duplicate date-station combinations ---
dupes_dwd = dwd[dwd.duplicated(subset=['date', 'station_name'], keep=False)]
print(f'Duplicate date-station rows in DWD: {len(dupes_dwd)}')
if len(dupes_dwd) > 0:
    print(dupes_dwd)

Duplicate date-station rows in DWD: 0


In [16]:
# --- On es concentren els nuls? ---
print('=== DISTRIBUCIÓ TEMPORAL DELS NULS PER ESTACIÓ ===')
for station, group in dwd.groupby("station_name"):
    nulls = group[group["wind_speed_ms"].isnull()]
    if len(nulls) > 0:
        print(f"\n{station} ({len(nulls)} nuls):")
        print(f"  Primer null : {nulls['date'].min().date()}")
        print(f"  Últim null  : {nulls['date'].max().date()}")
        print(f"  Per any:")
        print(nulls.groupby(nulls["date"].dt.year).size().to_string())
    else:
        print(f"\n{station}: cap null ✓")

=== DISTRIBUCIÓ TEMPORAL DELS NULS PER ESTACIÓ ===

Fehmarn (33 nuls):
  Primer null : 2021-08-23
  Últim null  : 2025-01-07
  Per any:
date
2021     2
2022     3
2024    26
2025     2

Helgoland (2 nuls):
  Primer null : 2022-01-10
  Últim null  : 2024-11-21
  Per any:
date
2022    1
2024    1

List auf Sylt (93 nuls):
  Primer null : 2021-01-05
  Últim null  : 2022-11-29
  Per any:
date
2021    41
2022    52


In [17]:
# --- Gap length analysis for null values at List auf Sylt ---
sylt = dwd[dwd["station_name"] == "List auf Sylt"].sort_values("date").copy()
sylt["is_null"] = sylt["wind_speed_ms"].isnull()
sylt["gap_id"] = (sylt["is_null"] != sylt["is_null"].shift()).cumsum()

gaps = (
    sylt[sylt["is_null"]]
    .groupby("gap_id")["date"]
    .agg(start="min", end="max", days="count")
    .sort_values("days", ascending=False)
)
print(f"Number of gaps: {len(gaps)}")
print(f"Longest gap   : {gaps['days'].max()} days")
print(f"Average gap   : {gaps['days'].mean():.1f} days")
print()
print(gaps.head(10))

Number of gaps: 4
Longest gap   : 51 days
Average gap   : 23.2 days

            start        end  days
gap_id                            
8      2022-10-10 2022-11-29    51
4      2021-01-12 2021-02-15    35
2      2021-01-05 2021-01-10     6
6      2022-03-22 2022-03-22     1


In [18]:
# --- Quick comparison Sylt vs Helgoland — year 2023 ---
comparison = (
    dwd[dwd["date"].dt.year == 2023]
    .groupby("station_name")["wind_speed_ms"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
        nulls=lambda x: x.isnull().sum()
    )
    .loc[["List auf Sylt", "Helgoland"]]
    .round(2)
)
print("=== Wind speed comparison 2023 (m/s) ===")
print(comparison)

=== Wind speed comparison 2023 (m/s) ===
               mean  median  std  min   max  nulls
station_name                                      
List auf Sylt  7.49    6.80 3.02 2.40 16.60      0
Helgoland      4.36    4.00 1.82 1.40 11.30      0


In [19]:
# --- Null treatment strategy for List auf Sylt ---
# Short gaps (<=7 days): linear interpolation
# Long gaps (>7 days) : proxy from Helgoland, scaled by mean ratio

dwd_clean = dwd.copy().sort_values(["station_name", "date"])

# Step 1 — linear interpolation for short gaps only
dwd_clean["wind_speed_ms"] = (
    dwd_clean
    .groupby("station_name")["wind_speed_ms"]
    .transform(lambda x: x.interpolate(method="linear", limit=7))
)

# Step 2 — compute scaling ratio Sylt / Helgoland (on days both have data)
helgoland = dwd_clean[dwd_clean["station_name"] == "Helgoland"][["date", "wind_speed_ms"]].rename(columns={"wind_speed_ms": "ws_helgoland"})
sylt_obs   = dwd_clean[(dwd_clean["station_name"] == "List auf Sylt") & dwd_clean["wind_speed_ms"].notna()][["date", "wind_speed_ms"]].rename(columns={"wind_speed_ms": "ws_sylt"})

ratio_df = sylt_obs.merge(helgoland, on="date")
scaling_ratio = (ratio_df["ws_sylt"] / ratio_df["ws_helgoland"]).mean()
print(f"Scaling ratio Sylt / Helgoland: {scaling_ratio:.4f}")

# Step 3 — fill remaining nulls at Sylt using scaled Helgoland
sylt_mask = (dwd_clean["station_name"] == "List auf Sylt") & dwd_clean["wind_speed_ms"].isna()
dwd_clean = dwd_clean.merge(helgoland, on="date", how="left")
dwd_clean.loc[sylt_mask, "wind_speed_ms"] = (
    dwd_clean.loc[sylt_mask, "ws_helgoland"] * scaling_ratio
)
dwd_clean = dwd_clean.drop(columns=["ws_helgoland"])

# Verification
remaining = dwd_clean["wind_speed_ms"].isnull().sum()
print(f"Remaining nulls after treatment: {remaining}")
print("All nulls resolved. ✓" if remaining == 0 else "⚠️  Some nulls remain — check manually.")

Scaling ratio Sylt / Helgoland: 1.7883
Remaining nulls after treatment: 6
⚠️  Some nulls remain — check manually.


In [20]:
# --- Inspect remaining nulls after treatment ---
remaining_nulls = dwd_clean[dwd_clean["wind_speed_ms"].isnull()]
print(f"Total remaining nulls: {len(remaining_nulls)}")
print()
print(remaining_nulls[["date", "station_name", "wind_speed_ms"]].to_string())

Total remaining nulls: 6

           date station_name  wind_speed_ms
1429 2024-12-05      Fehmarn            NaN
1430 2024-12-06      Fehmarn            NaN
1431 2024-12-07      Fehmarn            NaN
1432 2024-12-08      Fehmarn            NaN
1433 2024-12-09      Fehmarn            NaN
1434 2024-12-10      Fehmarn            NaN


In [21]:
# --- Fix remaining 6 nulls at Fehmarn (2024-12-05 to 2024-12-10) ---
# 6 consecutive days — linear interpolation is appropriate here.

dwd_clean["wind_speed_ms"] = (
    dwd_clean
    .groupby("station_name")["wind_speed_ms"]
    .transform(lambda x: x.interpolate(method="linear", limit=7))
)

# Final verification
remaining = dwd_clean["wind_speed_ms"].isnull().sum()
print(f"Remaining nulls: {remaining}")
print("Dataset clean. ✓" if remaining == 0 else "⚠️  Still some nulls — check manually.")

Remaining nulls: 0
Dataset clean. ✓


## 3.3 Cleaning Notes — List auf Sylt null treatment

Inspection revealed 4 gaps in the `wind_speed_ms` series at List auf Sylt,
all concentrated in 2021–2022:

| Gap | Start | End | Days | Treatment |
|-----|-------|-----|------|-----------|
| 1 | 2021-01-05 | 2021-01-10 | 6 | Linear interpolation |
| 2 | 2021-01-12 | 2021-02-15 | 35 | Helgoland proxy (scaled) |
| 3 | 2022-03-22 | 2022-03-22 | 1 | Linear interpolation |
| 4 | 2022-10-10 | 2022-11-29 | 51 | Helgoland proxy (scaled) |

Gaps of 35 and 51 days are too long for linear interpolation, which would
produce artificially smooth, unrealistic values over such a period.
Instead, Helgoland was used as a proxy station, as it is geographically
close and also located on the North Sea.

Because the two stations record systematically different wind speeds —
Sylt mean 7.49 m/s vs Helgoland mean 4.36 m/s in 2023 (ratio ≈ 1.72) —
Helgoland values were multiplied by a scaling ratio computed from all days
in which both stations had valid observations. This preserves the physical
characteristics of Sylt rather than underestimating wind speed during the
gap periods.

**Limitation:** the scaling ratio is assumed to be stable across time and
seasons. In reality, the ratio may vary month by month. For a production
model, a seasonally-adjusted ratio would be more appropriate.

Fehmarn had one additional gap of 6 consecutive days (2024-12-05 to
2024-12-10), treated with linear interpolation.

After all treatments, remaining nulls = 0. ✓

## 3.4 Cleaning Summary

| Check | SMARD | DWD |
|---|---|---|
| Null values | 0 | 128 → resolved ✓ |
| Missing dates | 0 | 0 |
| Negative values | 0 | 0 |
| Duplicates | 0 | 0 |
| Date range OK | 2021–2025 ✓ | 2021–2025 ✓ |

**DWD null treatment:**
- List auf Sylt (93 nulls, gaps up to 51 days) — short gaps (≤7 days): linear interpolation; long gaps (35 and 51 days): Helgoland proxy with scaling ratio ≈ 1.72
- Helgoland (2 nulls) — linear interpolation
- Fehmarn (38 nulls + 6 residual) — linear interpolation

All datasets clean and ready for merge. ✓

---
## 4. Merge

In [22]:
# --- Pivot DWD: one row per date, one column per station ---
dwd_pivot = dwd_clean.pivot_table(
    index="date",
    columns="station_name",
    values="wind_speed_ms"
).reset_index()

# Rename columns for clarity
dwd_pivot.columns.name = None
dwd_pivot = dwd_pivot.rename(columns={
    "Fehmarn"      : "wind_fehmarn_ms",
    "Helgoland"    : "wind_helgoland_ms",
    "List auf Sylt": "wind_sylt_ms"
})

print(f"Shape: {dwd_pivot.shape}")
print()
dwd_pivot.head()

Shape: (1826, 4)



,date,wind_fehmarn_ms,wind_helgoland_ms,wind_sylt_ms
0,2021-01-01,2.70,2.20,2.90
1,2021-01-02,2.80,2.70,4.20
2,2021-01-03,4.40,7.80,6.60
3,2021-01-04,6.40,9.40,7.80
4,2021-01-05,4.70,8.50,8.07


In [23]:
# --- INNER JOIN via SQL:
import sqlite3

conn = sqlite3.connect(":memory:")

smard.to_sql("smard", conn, index=False, if_exists="replace")
dwd_pivot.to_sql("dwd", conn, index=False, if_exists="replace")

query = """
    SELECT
        s.date,
        s.wind_offshore_mwh,
        d.wind_sylt_ms,
        d.wind_helgoland_ms,
        d.wind_fehmarn_ms
    FROM smard s
    INNER JOIN dwd d
        ON s.date = d.date
    ORDER BY s.date
"""

df = pd.read_sql_query(query, conn)
conn.close()

# Convert date back to datetime (sqlite returns it as string)
df["date"] = pd.to_datetime(df["date"])

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print()
df.head()

Shape: (1825, 5)
Date range: 2021-01-01 → 2025-12-30



,date,wind_offshore_mwh,wind_sylt_ms,wind_helgoland_ms,wind_fehmarn_ms
0,2021-01-01,17009.50,2.90,2.20,2.70
1,2021-01-02,102594.25,4.20,2.70,2.80
2,2021-01-03,121213.50,6.60,7.80,4.40
3,2021-01-04,139903.25,7.80,9.40,6.40
4,2021-01-05,87956.75,8.07,8.50,4.70


In [24]:
# --- Merge validation ---
expected_rows = len(smard)
print(f"SMARD rows        : {expected_rows}")
print(f"Merged rows       : {len(df)}")
print(f"Rows lost in join : {expected_rows - len(df)}")
print()
print(f"Null values in merged dataset:")
print(df.isnull().sum())
print()
print("Merge successful. ✓" if df.isnull().sum().sum() == 0 else "⚠️  Check nulls.")

SMARD rows        : 1825
Merged rows       : 1825
Rows lost in join : 0

Null values in merged dataset:
date                 0
wind_offshore_mwh    0
wind_sylt_ms         0
wind_helgoland_ms    0
wind_fehmarn_ms      5
dtype: int64

⚠️  Check nulls.


In [25]:
df.shape

(1825, 5)

In [26]:
# --- Inspect remaining nulls in wind_fehmarn_ms ---
print(df[df["wind_fehmarn_ms"].isnull()][["date", "wind_fehmarn_ms"]])

           date  wind_fehmarn_ms
1209 2024-04-24              NaN
1210 2024-04-25              NaN
1211 2024-04-26              NaN
1212 2024-04-27              NaN
1213 2024-04-28              NaN


In [27]:
# --- Fix 5 remaining nulls at Fehmarn (2024-04-24 to 2024-04-28) ---
df["wind_fehmarn_ms"] = df["wind_fehmarn_ms"].interpolate(method="linear", limit=7)

# Final verification
remaining = df.isnull().sum().sum()
print(f"Remaining nulls in merged dataset: {remaining}")
print("Merge complete. Dataset ready for EDA. ✓" if remaining == 0 else "⚠️  Still some nulls.")

Remaining nulls in merged dataset: 0
Merge complete. Dataset ready for EDA. ✓


In [28]:
# --- Save merged dataset to processed folder ---
df.to_csv("../data/processed/wind_merged_2021_2025.csv", index=False)

print("File saved: ../data/processed/wind_merged_2021_2025.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

File saved: ../data/processed/wind_merged_2021_2025.csv
Shape: (1825, 5)
Columns: ['date', 'wind_offshore_mwh', 'wind_sylt_ms', 'wind_helgoland_ms', 'wind_fehmarn_ms']


In [29]:
# --- Add sea-level aggregate columns ---
# North Sea : average of Sylt and Helgoland
# Baltic Sea : Fehmarn (only station available)
df["wind_north_sea_ms"] = df[["wind_sylt_ms", "wind_helgoland_ms"]].mean(axis=1)
df["wind_baltic_ms"]    = df["wind_fehmarn_ms"]

print("New columns added:")
print(f"  wind_north_sea_ms — mean of Sylt and Helgoland (North Sea)")
print(f"  wind_baltic_ms    — Fehmarn (Baltic Sea)")
print()
print(df[["date", "wind_offshore_mwh", "wind_north_sea_ms", "wind_baltic_ms"]].head())

New columns added:
  wind_north_sea_ms — mean of Sylt and Helgoland (North Sea)
  wind_baltic_ms    — Fehmarn (Baltic Sea)

        date  wind_offshore_mwh  wind_north_sea_ms  wind_baltic_ms
0 2021-01-01           17009.50               2.55            2.70
1 2021-01-02          102594.25               3.45            2.80
2 2021-01-03          121213.50               7.20            4.40
3 2021-01-04          139903.25               8.60            6.40
4 2021-01-05           87956.75               8.29            4.70


In [30]:
df.head()

,date,wind_offshore_mwh,wind_sylt_ms,wind_helgoland_ms,wind_fehmarn_ms,wind_north_sea_ms,wind_baltic_ms
0,2021-01-01,17009.50,2.90,2.20,2.70,2.55,2.70
1,2021-01-02,102594.25,4.20,2.70,2.80,3.45,2.80
2,2021-01-03,121213.50,6.60,7.80,4.40,7.20,4.40
3,2021-01-04,139903.25,7.80,9.40,6.40,8.60,6.40
4,2021-01-05,87956.75,8.07,8.50,4.70,8.29,4.70


In [31]:
# --- Rename SMARD column for clarity ---
df = df.rename(columns={"wind_offshore_mwh": "generation_offshore_mwh"})

# Overwrite processed CSV with updated column name
df.to_csv("../data/processed/wind_merged_2021_2025.csv", index=False)

print(f"Column renamed: wind_offshore_mwh → generation_offshore_mwh")
print(f"Columns: {list(df.columns)}")

Column renamed: wind_offshore_mwh → generation_offshore_mwh
Columns: ['date', 'generation_offshore_mwh', 'wind_sylt_ms', 'wind_helgoland_ms', 'wind_fehmarn_ms', 'wind_north_sea_ms', 'wind_baltic_ms']


Note: SMARD raw data uses "wind_offshore_mwh" 
— renamed here for clarity to distinguish energy generation (MWh) from wind speed measurements (m/s).

In [33]:
df.head()

,date,generation_offshore_mwh,wind_sylt_ms,wind_helgoland_ms,wind_fehmarn_ms,wind_north_sea_ms,wind_baltic_ms
0,2021-01-01,17009.50,2.90,2.20,2.70,2.55,2.70
1,2021-01-02,102594.25,4.20,2.70,2.80,3.45,2.80
2,2021-01-03,121213.50,6.60,7.80,4.40,7.20,4.40
3,2021-01-04,139903.25,7.80,9.40,6.40,8.60,6.40
4,2021-01-05,87956.75,8.07,8.50,4.70,8.29,4.70


---
## 5. EDA

In [34]:
# --- Station metadata ---
STATION_SEA = {
    "wind_sylt_ms"      : "North Sea",
    "wind_helgoland_ms" : "North Sea",
    "wind_fehmarn_ms"   : "Baltic Sea"
}

STATION_COORDS = {
    "wind_sylt_ms"      : (55.01, 8.41),
    "wind_helgoland_ms" : (54.18, 7.88),
    "wind_fehmarn_ms"   : (54.53, 11.06)
}

print("Station → Sea mapping:")
for station, sea in STATION_SEA.items():
    print(f"  {station:<22} → {sea}")

Station → Sea mapping:
  wind_sylt_ms           → North Sea
  wind_helgoland_ms      → North Sea
  wind_fehmarn_ms        → Baltic Sea


---
## 6. Correlation Analysis *(pròxima sessió)*